In [1]:
import numpy as np
import yaml
from tqdm import tqdm

from babybench import utils as bb_utils
from mimoProprioception.proprio import SimpleProprioception
from modeling import MonteCarloCBN
from policies import CBNExplorationPolicy
from utils.env import variable_filter, variable_list

In [2]:
env_config = './configs/config_proprio.yml'

pretrain_data = './data/rnd_proprio_spring-act_steps-1k_fskip-50.pkl'
cbn_model = './models/mc-cbn_proprio_gmd-noise_N=1k_fskip=50_lim-inferred.pkl'

with open(env_config) as f:
    config = yaml.safe_load(f)

In [3]:
env = bb_utils.make_env(config)

In [4]:
n_steps = 10_000
anatomy = ('shoulder', 'elbow', 'hand', 'fingers')

act_names = [
    act_name
    for act_idx in range(env.model.nu)
    if (act_name := env.model.actuator(act_idx).name).startswith('act:')
]
n_actuators = env.model.nu

joint_names = env.proprioception.joint_names
rel_joint_ids, rel_joint_names = zip(*[
    (joint_id, joint_name.removeprefix('robot:'))
    for joint_id, joint_name in enumerate(joint_names)
    for part in anatomy
    if part in joint_name
])

state_vars = variable_filter(
    var_list=variable_list(pretrain_data),
    pattern=r'joint:'
)

In [5]:
def filter_proprio(
        proprio: SimpleProprioception,
        relevant_joints: list[int]
) -> np.ndarray:
    sensors: dict[str, np.ndarray] = proprio.sensor_outputs
    qpos = sensors['qpos']

    qpos_red = np.array([
        qpos[joint_id]
        for joint_id in relevant_joints
    ])

    return qpos_red

In [6]:
cbn = MonteCarloCBN(load_file=cbn_model)

policy = CBNExplorationPolicy(
    cbn=cbn,

    action_vars=act_names,
    state_vars=state_vars,
    sampled_state_vars=variable_filter(
        var_list=state_vars,
        pattern=r'.+\.pos1'
    ),
    fixed_state_vars=variable_filter(
        var_list=state_vars,
        pattern=r'.+\.pos0'
    ),

    history_limit=50,
    expl_limit=10,
    expl_init_delta=0.1,
    expl_delta_gain=1.25,
    min_results=25,
    target_thresh=1e-2,

    verbose=True
)

In [7]:
env.reset()

qpos0_vec = filter_proprio(
    proprio=env.proprioception,
    relevant_joints=rel_joint_ids
)
s0 = {
    f'joint:{j_name}.pos0': qpos0_vec[idx]
    for idx, j_name in enumerate(rel_joint_names)
}

env.reset()

qpos1_vec = filter_proprio(
    proprio=env.proprioception,
    relevant_joints=rel_joint_ids
)
s1 = {
    f'joint:{j_name}.pos1': qpos1_vec[idx]
    for idx, j_name in enumerate(rel_joint_names)
}

dpos_vec: np.ndarray = qpos1_vec - qpos0_vec
ds = {
    f'joint:{j_name}.dpos': dpos_vec[idx]
    for idx, j_name in enumerate(rel_joint_names)
}

state = s0 | s1 | ds

In [ ]:
imgs: list[np.ndarray] = []

for _ in tqdm(range(n_steps), desc='Step', unit_scale=True):
    action = policy(state)
    obs, _, terminated, truncated, info = env.step(action)

    qpos0_vec = qpos1_vec.copy()
    s0 = {
        f'joint:{j_name}.pos0': qpos0_vec[idx]
        for idx, j_name in enumerate(rel_joint_names)
    }

    qpos1_vec = filter_proprio(
        proprio=env.proprioception,
        relevant_joints=rel_joint_ids
    )
    s1 = {
        f'joint:{j_name}.pos1': qpos1_vec[idx]
        for idx, j_name in enumerate(rel_joint_names)
    }

    dpos_vec: np.ndarray = qpos1_vec - qpos0_vec
    ds = {
        f'joint:{j_name}.dpos': dpos_vec[idx]
        for idx, j_name in enumerate(rel_joint_names)
    }

    state = s0 | s1 | ds

    imgs.append(bb_utils.render(env))

    if terminated or truncated:
        env.reset()

env.close()

Step:   0%|          | 0.00/10.0k [00:00<?, ?it/s]/home/mcibula/miniconda3/envs/babybench/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/mcibula/miniconda3/envs/babybench/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Starting new exploration episode...
Starting backpropagated rejection sampling with delta = 0.10
Iteration 1 (delta = 0.10): Failed. Found 4 valid samples, repeating.
Iteration 2 (delta = 0.12): Failed. Found 18 valid samples, repeating.
Iteration 3 (delta = 0.16): Failed. Found 8 valid samples, repeating.
Iteration 4 (delta = 0.20): Failed. Found 19 valid samples, repeating.
Iteration 5 (delta = 0.24): Failed. Found 4 valid samples, repeating.
Iteration 6 (delta = 0.31): Failed. Found 7 valid samples, repeating.
Iteration 7 (delta = 0.38): Failed. Found 4 valid samples, repeating.
Iteration 8 (delta = 0.48): Failed. Found 10 valid samples, repeating.
Iteration 9 (delta = 0.60): Failed. Found 5 valid samples, repeating.
Iteration 10 (delta = 0.75): Failed. Found 20 valid samples, repeating.
Iteration 11 (delta = 0.93): Failed. Found 16 valid samples, repeating.
Iteration 12 (delta = 1.16): Failed. Found 16 valid samples, repeating.
Iteration 13 (delta = 1.46): Success. Found 104 valid 

Step:   0%|          | 1.00/10.0k [00:02<5:42:41, 2.06s/it]

Starting backpropagated rejection sampling with delta = 0.10
Iteration 1 (delta = 0.10): Failed. Found 4 valid samples, repeating.
Iteration 2 (delta = 0.12): Failed. Found 18 valid samples, repeating.
Iteration 3 (delta = 0.16): Failed. Found 8 valid samples, repeating.
Iteration 4 (delta = 0.20): Failed. Found 19 valid samples, repeating.
Iteration 5 (delta = 0.24): Failed. Found 4 valid samples, repeating.
Iteration 6 (delta = 0.31): Failed. Found 7 valid samples, repeating.
Iteration 7 (delta = 0.38): Failed. Found 4 valid samples, repeating.
Iteration 8 (delta = 0.48): Failed. Found 10 valid samples, repeating.
Iteration 9 (delta = 0.60): Failed. Found 5 valid samples, repeating.
Iteration 10 (delta = 0.75): Failed. Found 20 valid samples, repeating.
Iteration 11 (delta = 0.93): Failed. Found 16 valid samples, repeating.
Iteration 12 (delta = 1.16): Failed. Found 16 valid samples, repeating.
Iteration 13 (delta = 1.46): 

Step:   0%|          | 2.00/10.0k [00:02<3:27:13, 1.24s/it]

...

Success. Found 60 valid samples.
Starting backpropagated rejection sampling with delta = 0.10
Iteration 1 (delta = 0.10): Failed. Found 1 valid samples, repeating.
Iteration 2 (delta = 0.12): Failed. Found 5 valid samples, repeating.
Iteration 3 (delta = 0.16): Failed. Found 21 valid samples, repeating.
Iteration 4 (delta = 0.20): Failed. Found 6 valid samples, repeating.
Iteration 5 (delta = 0.24): Failed. Found 6 valid samples, repeating.
Iteration 6 (delta = 0.31): Failed. Found 23 valid samples, repeating.
Iteration 7 (delta = 0.38): Failed. Found 14 valid samples, repeating.
Iteration 8 (delta = 0.48): Failed. Found 19 valid samples, repeating.
Iteration 9 (delta = 0.60): Failed. Found 11 valid samples, repeating.
Iteration 10 (delta = 0.75): Failed. Found 19 valid samples, repeating.
Iteration 11 (delta = 0.93): Failed. Found 15 valid samples, repeating.
Iteration 12 (delta = 1.16): Failed. Found 14 valid samples, repeating.
Iteration 13 (delta = 1.46): 

Step:  10%|▉         | 987/10.0k [17:26<2:03:51, 1.21it/s]

Success. Found 76 valid samples.
Starting backpropagated rejection sampling with delta = 0.10
Iteration 1 (delta = 0.10): Failed. Found 1 valid samples, repeating.
Iteration 2 (delta = 0.12): Failed. Found 5 valid samples, repeating.
Iteration 3 (delta = 0.16): Failed. Found 21 valid samples, repeating.
Iteration 4 (delta = 0.20): Failed. Found 6 valid samples, repeating.
Iteration 5 (delta = 0.24): Failed. Found 6 valid samples, repeating.
Iteration 6 (delta = 0.31): Failed. Found 23 valid samples, repeating.
Iteration 7 (delta = 0.38): Failed. Found 14 valid samples, repeating.
Iteration 8 (delta = 0.48): Failed. Found 19 valid samples, repeating.
Iteration 9 (delta = 0.60): Failed. Found 11 valid samples, repeating.
Iteration 10 (delta = 0.75): Failed. Found 19 valid samples, repeating.
Iteration 11 (delta = 0.93): Failed. Found 21 valid samples, repeating.
Iteration 12 (delta = 1.16): Failed. Found 10 valid samples, repeating.
Iteration 13 (delta = 1.46): 

Step:  10%|▉         | 988/10.0k [17:27<2:00:04, 1.25it/s]

Success. Found 191 valid samples.
Starting backpropagated rejection sampling with delta = 0.10
Iteration 1 (delta = 0.10): Failed. Found 1 valid samples, repeating.
Iteration 2 (delta = 0.12): Failed. Found 5 valid samples, repeating.
Iteration 3 (delta = 0.16): Failed. Found 21 valid samples, repeating.
Iteration 4 (delta = 0.20): Failed. Found 6 valid samples, repeating.
Iteration 5 (delta = 0.24): Failed. Found 6 valid samples, repeating.
Iteration 6 (delta = 0.31): Failed. Found 23 valid samples, repeating.
Iteration 7 (delta = 0.38): Failed. Found 14 valid samples, repeating.
Iteration 8 (delta = 0.48): Failed. Found 19 valid samples, repeating.
Iteration 9 (delta = 0.60): Failed. Found 11 valid samples, repeating.
Iteration 10 (delta = 0.75): Failed. Found 19 valid samples, repeating.
Iteration 11 (delta = 0.93): 

In [ ]:
video_path = './results/videos/cbn_proprio_fskip=50_1k-pretrain_10k-active_history-limit=50.mp4'

bb_utils.evaluation_video(
    images=imgs,
    save_name=video_path,
    resolution=(480, 480)
)